# Notebook 02g — Multiclass ViT-B/16
**D7047E Advanced Deep Learning | Group 14**

Task: Multiclass — 8 classes (CLEAN + 7 cross-out styles)  
Model: ViT-B/16 (frozen backbone + new head)  

WandB: `adl-crossouts-v3 / multiclass / ViT-B/16`

## 1. Configuration

In [ ]:
import sys
sys.path.insert(0, '..')
DATA_DIR       = '../dataset/iam_crossouts'
CHECKPOINT_DIR = '../checkpoints'
ZIP_PATH       = '../dataset/adl_dataset.zip'
FILE_ID        = '1dgIfz8aFwCuphLN9-L7gcU4QN0UQEKP3'
IMG_SIZE       = 224
BATCH_SIZE     = 256
NUM_WORKERS    = 16
LR             = 4e-4
EPOCHS         = 100
MIN_EPOCHS     = 20
PATIENCE       = 15
MODEL          = 'ViT-B/16'
from common import WANDB_GROUP_MULTICLASS
WANDB_GROUP    = WANDB_GROUP_MULTICLASS
print(f'Config loaded. Training: {MODEL}')

## 2. Setup

In [ ]:
!pip install -q gdown torch torchvision pillow matplotlib scikit-learn wandb python-dotenv

In [ ]:
import os, zipfile, gc
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'
import torch, torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import models
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             classification_report, confusion_matrix, ConfusionMatrixDisplay)
import wandb
from dotenv import load_dotenv

from common import get_transforms, WANDB_PROJECT, WANDB_GROUP_BINARY, WANDB_GROUP_MULTICLASS, log_test_metrics, CrossOutDataset, train_model, CATEGORIES, log_confusion_matrix

WANDB_GROUP = WANDB_GROUP_MULTICLASS

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
load_dotenv()
wandb.login(key=os.environ.get('WANDB_API_KEY'))
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## 3. Dataset Download

In [ ]:
try:
    import gdown
except ImportError:
    import subprocess; subprocess.run(['pip','install','gdown','-q'],check=True); import gdown
os.makedirs(os.path.dirname(ZIP_PATH), exist_ok=True)
if not os.path.exists(ZIP_PATH):
    gdown.download(f'https://drive.google.com/uc?id={FILE_ID}', ZIP_PATH, quiet=False)
if not os.path.exists(DATA_DIR):
    with zipfile.ZipFile(ZIP_PATH,'r') as zf: zf.extractall(DATA_DIR)
print('Dataset ready.')

## 4. Data Loaders

In [ ]:
train_t, val_t = get_transforms(IMG_SIZE)
NC = len(CATEGORIES)

mc_train = CrossOutDataset(os.path.join(DATA_DIR,'train','images'), CATEGORIES, train_t)
mc_val   = CrossOutDataset(os.path.join(DATA_DIR,'val',  'images'), CATEGORIES, val_t)
mc_test  = CrossOutDataset(os.path.join(DATA_DIR,'test', 'images'), CATEGORIES, val_t)

ldr_kw = dict(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
              pin_memory=True, prefetch_factor=2 if NUM_WORKERS>0 else None)
train_loader = DataLoader(mc_train, shuffle=True,  **ldr_kw)
val_loader   = DataLoader(mc_val,   shuffle=False, **ldr_kw)
test_loader  = DataLoader(mc_test,  shuffle=False, **ldr_kw)
print(f'Train: {len(mc_train):,}  Val: {len(mc_val):,}  Test: {len(mc_test):,}')

## 5. Model Builder
Backbone frozen. Only the classification head is trained.

In [ ]:
def build_vit():
    m = models.vit_b_16(weights=models.ViT_B_16_Weights.IMAGENET1K_V1)
    for p in m.parameters(): p.requires_grad = False
    m.heads = nn.Linear(m.heads.head.in_features, NC)
    return m

model = build_vit()
tr  = sum(p.numel() for p in model.parameters() if p.requires_grad)
tot = sum(p.numel() for p in model.parameters())
print(f'ViT-B/16  trainable: {tr:,} / {tot:,}')

## 6. Train

In [ ]:
safe      = MODEL.replace('/','_').replace('-','_')
save_path = os.path.join(CHECKPOINT_DIR, f'best_mc_{safe}.pth')
resume_path = save_path if os.path.exists(save_path) else None
if resume_path: print(f'Resuming from {resume_path}')

trained_model, history = train_model(
    MODEL, model, train_loader, val_loader,
    nn.CrossEntropyLoss(),
    task='multiclass', save_path=save_path, device=device,
    lr=LR, epochs=EPOCHS, min_epochs=MIN_EPOCHS, patience=PATIENCE,
    wandb_project=WANDB_PROJECT, wandb_group=WANDB_GROUP, batch_size=BATCH_SIZE,
    resume_path=resume_path,
)
del trained_model; torch.cuda.empty_cache(); gc.collect()

## 7. Test Evaluation

In [ ]:
ckpt = torch.load(save_path, map_location=device)
model.load_state_dict(ckpt['model_state_dict']); model.to(device); model.eval()

preds, labels = [], []
with torch.no_grad():
    for imgs, lbs in test_loader:
        preds.extend(model(imgs.to(device)).argmax(1).cpu().tolist())
        labels.extend(lbs.tolist())

print(f'Multiclass — {MODEL} — Test Results')
print(f'  Accuracy:  {accuracy_score(labels, preds):.4f}')
print(f'  Macro-P:   {precision_score(labels, preds, average="macro", zero_division=0):.4f}')
print(f'  Macro-R:   {recall_score(labels, preds, average="macro", zero_division=0):.4f}')
print(f'  Macro-F1:  {f1_score(labels, preds, average="macro", zero_division=0):.4f}')
print()
print(classification_report(labels, preds, target_names=CATEGORIES))

In [ ]:
log_test_metrics(
    preds=preds, labels=labels,
    
    model_name=MODEL,
    wandb_project=WANDB_PROJECT,
    wandb_group=WANDB_GROUP,
    task='multiclass',
)


## 8. Confusion Matrix

In [ ]:
cm = confusion_matrix(labels, preds)
fig, ax = plt.subplots(figsize=(10, 8))
ConfusionMatrixDisplay(cm, display_labels=CATEGORIES).plot(
    ax=ax, xticks_rotation=45, colorbar=False, cmap='Blues')
plt.title(f'Confusion Matrix — {MODEL}')
plt.tight_layout()
safe = MODEL.replace('/','_').replace('-','_')
plt.savefig(f'mc_{safe}_cm.png', dpi=150); plt.show()
print(f'Saved: mc_{safe}_cm.png')

In [ ]:
from common import log_confusion_matrix
log_confusion_matrix(
    preds=preds, labels=labels,
    class_names=CATEGORIES,
    model_name=MODEL,
    wandb_project=WANDB_PROJECT,
    wandb_group=WANDB_GROUP,
)
print('Confusion matrix logged to WandB.')

## 9. Training Curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history['train_loss'],label='Train'); ax1.plot(history['val_loss'],label='Val')
ax1.set_title(f'{MODEL} Multiclass — Loss'); ax1.set_xlabel('Epoch'); ax1.legend()
ax2.plot(history['train_acc'],label='Train'); ax2.plot(history['val_acc'],label='Val')
ax2.set_title(f'{MODEL} Multiclass — Accuracy'); ax2.set_xlabel('Epoch'); ax2.legend()
plt.tight_layout()
safe = MODEL.replace('/','_').replace('-','_')
plt.savefig(f'mc_{safe}_curves.png', dpi=150); plt.show()
print(f'Saved: mc_{safe}_curves.png')
del model; torch.cuda.empty_cache(); gc.collect()